# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. All dataset elements such as record sets, fields, and columns are referenced by their `@id`.

### Dataset Source
The dataset is described in a Croissant schema available here:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Make sure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not subscript the metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview

List all available record sets, their `@id`, and fields with their `@id`. This provides an overview of how the data is structured in the Croissant schema.

In [ ]:
# Gather all record sets by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found. Please check the dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        # List fields for each RecordSet
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            # Each field is a dict with '@id'
            print(f"    Field: {field['@id']}")

## 3. Data Extraction

Load tabular data from each record set into pandas DataFrames for further analysis. We'll use the `@id` of each record set and its fields.

In [ ]:
# Gather all record set @id(s)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for RecordSet: {record_set_id}")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head(3))  # Show top 3 rows of each DataFrame

# For further EDA, select the main record set if found
if dataframes:
    # Choose the first record set for demonstration
    main_record_set_id = list(dataframes.keys())[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Grouping and aggregating data

We will demonstrate with a numeric column, referencing columns and fields by their `@id`.

In [ ]:
# EDA demonstration
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Main DataFrame columns: {df.columns.tolist()}")

    # Try to select a numeric field (heuristic: integer/float dtype or contains 'age' or 'interval' in column name)
    numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_candidates:
        # Try string-matching for likely numeric fields
        for col in df.columns:
            if any(s in col.lower() for s in ["age", "interval", "count", "metastasis"]):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notna().any():
                        numeric_candidates.append(col)
                except Exception:
                    continue
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")
        # Set an example threshold: mean or median
        try:
            threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        except Exception:
            threshold = 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head(5))

        # Normalize the selected field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nFirst 5 normalized values for {numeric_field_id} (by @id):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field (string/object dtype)
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        group_field = None
        for cand in group_candidates:
            if 'sex' in cand.lower() or 'type' in cand.lower() or 'status' in cand.lower():
                group_field = cand
                break
        if not group_field and group_candidates:
            group_field = group_candidates[0]

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].agg(['mean', 'count'])
            print(f"\nGrouped filtered data by {group_field} (showing mean/count):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for this record set.")
else:
    print("No main record set loaded for EDA section.")

## 5. Visualization

Visualize the distribution of a numeric field and its breakdown by a group field (referenced by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet: {main_record_set_id}")
    plt.show()
    
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field or group available for plotting.")

## 6. Conclusion

- Loaded metadata and tabular records from the FAIR² dataset using `mlcroissant`.
- Identified the structure: record sets, fields (`@id`), and available columns.
- Conducted exploratory analysis with normalization and grouping using only `@id` references.
- Visualized key numeric distributions and categorical groupings.

For more advanced tasks, adapt this workflow to prepare features, address clinical research questions, or build ML pipelines leveraging the FAIR-compliant Croissant metadata. All operations in this notebook used only the official public `@id` references for data transparency and reproducibility.